In [1]:
import collections
from pathlib import Path
from tqdm.notebook import tqdm  # ← notebook version looks nicer in Kaggle

def build_vocab_from_dataset(root_dir, min_count=2):
    files = list(Path(root_dir).rglob("*.agnostic"))
    print(f"Found {len(files):,} .agnostic files. Starting scan...\n")

    counter = collections.Counter()
    for i, path in enumerate(tqdm(files, desc="Building vocab", unit="file")):
        with open(path, encoding="utf-8", errors="ignore") as f:
            counter.update(f.read().split())

        if (i + 1) % 5000 == 0:
            print(f"  [{i+1:,} / {len(files):,}] — {len(counter):,} unique tokens so far")

    vocab = ["<blank>", "<unk>"]
    for token, count in counter.most_common():
        if count >= min_count:
            vocab.append(token)

    print(f"\nVocab size: {len(vocab)} tokens")
    print(f"Dropped (count=1): {sum(1 for c in counter.values() if c < min_count)} tokens")
    return vocab, counter

vocab, counter = build_vocab_from_dataset("/kaggle/input/datasets/piamac/cameraprimus/CameraPrIMuS - Copy")

print("\nVOCAB = [")
for t in vocab:
    print(f'    "{t}",')
print("]")

Found 3,996 .agnostic files. Starting scan...



Building vocab:   0%|          | 0/3996 [00:00<?, ?file/s]


Vocab size: 513 tokens
Dropped (count=1): 71 tokens

VOCAB = [
    "<blank>",
    "<unk>",
    "barline-L1",
    "clef.G-L2",
    "note.quarter-L4",
    "rest.eighth-L3",
    "note.quarter-S3",
    "rest.quarter-L3",
    "note.quarter-L3",
    "note.quarter-S4",
    "metersign.C-L3",
    "note.quarter-S2",
    "dot-S3",
    "note.eighth-L4",
    "dot-S4",
    "note.eighth-S3",
    "note.eighth-S4",
    "note.quarter-L5",
    "note.quarter-L2",
    "note.eighth-L3",
    "accidental.flat-L3",
    "note.eighth-S2",
    "accidental.sharp-L5",
    "multirest-L3",
    "clef.C-L1",
    "digit.4-L2",
    "dot-S2",
    "note.eighth-L5",
    "digit.3-L4",
    "note.eighth-L2",
    "note.quarter-S5",
    "dot-S5",
    "metersign.C/-L3",
    "rest.half-L3",
    "note.half-L4",
    "accidental.sharp-S3",
    "note.half-S3",
    "note.quarter-S1",
    "accidental.flat-S4",
    "note.beamedRight1-L4",
    "note.eighth-S5",
    "note.half-L3",
    "note.beamedRight1-S3",
    "note.half-S4",
    "note

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Vocabulary ────────────────────────────────────────────────────────────────
VOCAB = [
    "<blank>",
    "<unk>",
    "barline-L1",
    "clef.G-L2",
    "note.quarter-L4",
    "rest.eighth-L3",
    "note.quarter-S3",
    "rest.quarter-L3",
    "note.quarter-L3",
    "note.quarter-S4",
    "metersign.C-L3",
    "note.quarter-S2",
    "dot-S3",
    "note.eighth-L4",
    "dot-S4",
    "note.eighth-S3",
    "note.eighth-S4",
    "note.quarter-L5",
    "note.quarter-L2",
    "note.eighth-L3",
    "accidental.flat-L3",
    "note.eighth-S2",
    "accidental.sharp-L5",
    "multirest-L3",
    "clef.C-L1",
    "digit.4-L2",
    "dot-S2",
    "note.eighth-L5",
    "digit.3-L4",
    "note.eighth-L2",
    "note.quarter-S5",
    "dot-S5",
    "metersign.C/-L3",
    "rest.half-L3",
    "note.half-L4",
    "accidental.sharp-S3",
    "note.half-S3",
    "note.quarter-S1",
    "accidental.flat-S4",
    "note.beamedRight1-L4",
    "note.eighth-S5",
    "note.half-L3",
    "note.beamedRight1-S3",
    "note.half-S4",
    "note.half-S2",
    "note.quarter-L6",
    "note.beamedRight1-L3",
    "note.beamedRight1-S4",
    "accidental.flat-L4",
    "digit.1-S5",
    "accidental.sharp-S2",
    "note.beamedLeft2-L4",
    "note.beamedLeft2-S3",
    "note.beamedBoth2-L4",
    "digit.2-L4",
    "note.eighth-S1",
    "note.quarter-L1",
    "note.beamedLeft1-L4",
    "note.beamedRight1-L5",
    "dot-S1",
    "note.half-L2",
    "note.half-L5",
    "note.beamedLeft1-S3",
    "rest.sixteenth-L3",
    "note.beamedBoth2-S4",
    "note.beamedRight1-S2",
    "note.beamedLeft1-S4",
    "note.beamedLeft1-L3",
    "note.beamedLeft1-S2",
    "note.beamedLeft2-S4",
    "note.beamedLeft2-L3",
    "digit.8-L2",
    "accidental.flat-S2",
    "note.beamedBoth2-L3",
    "accidental.flat-L2",
    "note.beamedBoth2-L5",
    "note.beamedBoth2-S3",
    "dot-S6",
    "note.eighth-L6",
    "note.beamedLeft2-L5",
    "note.quarter-S0",
    "note.beamedRight2-L4",
    "note.beamedRight1-S5",
    "note.beamedRight1-L2",
    "note.beamedRight2-S4",
    "note.beamedRight2-S3",
    "note.beamedLeft2-S2",
    "note.beamedBoth2-S2",
    "digit.2-L2",
    "accidental.flat-S3",
    "note.beamedLeft1-L5",
    "clef.F-L4",
    "note.quarter-S6",
    "note.beamedBoth2-S5",
    "note.sixteenth-S3",
    "note.beamedLeft2-L2",
    "digit.2-S5",
    "accidental.sharp-L4",
    "note.beamedBoth1-S3",
    "note.beamedLeft1-L2",
    "note.half-S5",
    "note.beamedRight2-L3",
    "note.beamedLeft2-S5",
    "note.beamedRight2-L5",
    "note.beamedBoth1-L4",
    "accidental.sharp-S5",
    "note.beamedBoth1-S4",
    "note.beamedRight1-L6",
    "note.eighth-L1",
    "accidental.sharp-L1",
    "note.half-S1",
    "note.beamedBoth2-L2",
    "note.beamedBoth1-L3",
    "note.beamedLeft1-S5",
    "note.beamedRight1-S1",
    "note.sixteenth-L4",
    "note.sixteenth-S4",
    "note.beamedRight2-S2",
    "note.beamedBoth1-S2",
    "digit.6-L4",
    "note.beamedRight2-S5",
    "note.beamedBoth1-L5",
    "accidental.sharp-L3",
    "note.beamedBoth2-L6",
    "note.sixteenth-L3",
    "note.eighth-S0",
    "note.beamedBoth1-L2",
    "note.beamedLeft1-S1",
    "accidental.sharp-S4",
    "dot-S0",
    "note.sixteenth-L5",
    "note.beamedLeft2-L6",
    "note.half-L6",
    "note.beamedBoth2-S1",
    "fermata.above-S6",
    "note.beamedRight2-L2",
    "note.eighth-S6",
    "note.beamedLeft1-L1",
    "note.half-L1",
    "note.beamedRight1-L1",
    "note.beamedLeft2-S1",
    "note.sixteenth-S2",
    "note.quarter-L0",
    "note.beamedBoth1-S1",
    "accidental.sharp-L2",
    "accidental.natural-L3",
    "accidental.sharp-S1",
    "note.beamedRight1-S0",
    "note.beamedBoth1-S5",
    "note.whole-S3",
    "clef.C-L3",
    "digit.3-S5",
    "note.beamedLeft1-L6",
    "note.whole-L3",
    "note.beamedRight2-L6",
    "note.beamedRight1-S6",
    "note.whole-L4",
    "note.sixteenth-L2",
    "note.beamedBoth2-L1",
    "clef.C-L4",
    "digit.4-S5",
    "note.half-S0",
    "note.beamedLeft2-L1",
    "note.beamedBoth1-S0",
    "note.beamedRight2-S1",
    "note.beamedBoth2-S6",
    "note.beamedBoth1-L1",
    "note.beamedLeft2-S6",
    "slur.start-L4",
    "slur.end-L4",
    "note.beamedBoth2-S0",
    "note.beamedLeft1-S6",
    "note.whole-S2",
    "slur.start-S4",
    "slur.end-S4",
    "note.quarter-L7",
    "gracenote.eighth-S4",
    "note.beamedRight2-L1",
    "accidental.natural-L4",
    "note.sixteenth-S5",
    "accidental.natural-S3",
    "accidental.natural-S4",
    "gracenote.eighth-L4",
    "gracenote.eighth-S3",
    "note.beamedBoth3-L4",
    "note.whole-L2",
    "slur.start-L5",
    "slur.end-L5",
    "note.beamedBoth1-L6",
    "note.beamedBoth3-S3",
    "accidental.flat-S5",
    "note.quarter-S7",
    "note.half-S6",
    "note.beamedLeft3-S3",
    "digit.6-S5",
    "note.whole-S4",
    "gracenote.eighth-L5",
    "note.beamedLeft3-L4",
    "slur.start-S3",
    "slur.end-S3",
    "accidental.flat-L5",
    "note.beamedLeft2-S0",
    "note.beamedBoth3-L3",
    "digit.7-S5",
    "digit.5-S5",
    "note.beamedBoth3-S4",
    "note.beamedRight2-S6",
    "note.whole-L5",
    "digit.8-S5",
    "note.beamedLeft3-L3",
    "gracenote.eighth-S5",
    "note.beamedBoth2-L0",
    "note.beamedLeft1-S0",
    "note.beamedBoth3-S2",
    "gracenote.eighth-L3",
    "note.sixteenth-S1",
    "note.beamedRight2-S0",
    "note.beamedBoth1-S6",
    "note.beamedBoth3-L5",
    "note.eighth-L0",
    "accidental.natural-S2",
    "slur.start-L3",
    "slur.end-L3",
    "digit.0-S5",
    "note.quarter-S-1",
    "rest.whole-L4",
    "note.sixteenth-L6",
    "note.beamedRight1-L0",
    "note.beamedRight1-L7",
    "note.beamedLeft3-L5",
    "dot-S7",
    "slur.start-S5",
    "slur.end-S5",
    "note.whole-S1",
    "digit.4-L4",
    "digit.9-S5",
    "accidental.natural-S5",
    "gracenote.eighth-S2",
    "note.beamedLeft3-S4",
    "note.beamedBoth2-L7",
    "note.beamedRight3-L4",
    "note.sixteenth-L1",
    "note.beamedBoth3-S5",
    "note.beamedLeft2-L0",
    "note.half-L0",
    "slur.start-S2",
    "slur.end-S2",
    "note.beamedLeft2-L7",
    "note.beamedLeft3-S2",
    "note.beamedRight3-S2",
    "accidental.natural-L5",
    "accidental.natural-L2",
    "slur.start-L2",
    "slur.end-L2",
    "accidental.flat-S1",
    "note.beamedRight3-S3",
    "note.beamedLeft1-L0",
    "note.beamedBoth1-L0",
    "note.quarter-L-1",
    "dot-S-1",
    "note.beamedRight3-L3",
    "note.eighth-L7",
    "note.beamedLeft3-L2",
    "note.eighth-S7",
    "note.beamedRight1-S7",
    "note.beamedBoth3-L2",
    "clef.G-L1",
    "note.beamedBoth3-L6",
    "note.beamedLeft2-S-1",
    "note.beamedRight2-L0",
    "note.beamedLeft3-S5",
    "gracenote.sixteenth-S3",
    "accidental.sharp-L6",
    "note.whole-S5",
    "note.beamedLeft1-S-1",
    "note.beamedRight3-L5",
    "note.beamedRight2-L7",
    "note.beamedBoth3-L1",
    "note.whole-L1",
    "note.beamedLeft1-L7",
    "gracenote.sixteenth-L4",
    "gracenote.sixteenth-S4",
    "digit.12-L4",
    "note.eighth-S-1",
    "gracenote.eighth-L6",
    "slur.start-L6",
    "slur.end-L6",
    "gracenote.sixteenth-L5",
    "note.beamedRight1-S-1",
    "note.beamedLeft3-L1",
    "note.whole-S0",
    "note.beamedLeft3-L6",
    "note.beamedBoth1-L-1",
    "note.sixteenth-S0",
    "note.beamedRight3-S1",
    "note.beamedBoth2-S7",
    "accidental.natural-S1",
    "gracenote.sixteenth-L3",
    "note.beamedLeft3-S1",
    "rest.thirty_second-L3",
    "note.beamedBoth3-S1",
    "note.beamedBoth1-L7",
    "note.beamedRight3-S4",
    "note.beamedBoth3-S6",
    "note.beamedLeft1-L-1",
    "slur.start-S6",
    "slur.end-S6",
    "gracenote.eighth-L2",
    "note.beamedRight2-S7",
    "note.beamedBoth1-S-1",
    "note.beamedRight3-L2",
    "accidental.flat-L1",
    "note.half-S7",
    "note.sixteenth-S6",
    "gracenote.eighth-S1",
    "accidental.sharp-S0",
    "note.beamedBoth2-S-1",
    "accidental.natural-L1",
    "accidental.sharp-L0",
    "gracenote.sixteenth-S2",
    "accidental.flat-L6",
    "note.beamedRight1-L-1",
    "note.beamedLeft3-L0",
    "slur.start-S1",
    "slur.end-S1",
    "gracenote.sixteenth-L2",
    "note.half-L7",
    "note.beamedLeft1-S7",
    "note.eighth-L-1",
    "note.beamedLeft3-S6",
    "accidental.natural-S-1",
    "slur.start-S0",
    "slur.end-S0",
    "gracenote.sixteenth-S5",
    "note.quarter-S-2",
    "note.beamedLeft3-S0",
    "note.beamedBoth3-S0",
    "note.beamedLeft1-S-2",
    "clef.C-L2",
    "slur.start-L1",
    "slur.end-L1",
    "note.half-S-1",
    "accidental.natural-L6",
    "note.whole-L0",
    "note.beamedBoth1-S-2",
    "note.beamedBoth3-S-1",
    "digit.9-L4",
    "accidental.flat-S6",
    "note.beamedLeft2-S7",
    "accidental.flat-S-1",
    "note.beamedBoth3-L7",
    "slur.start-L7",
    "slur.end-L7",
    "note.beamedRight3-S6",
    "digit.1-L2",
    "accidental.natural-S6",
    "note.sixteenth-S7",
    "note.beamedRight2-S-1",
    "note.beamedLeft3-S-1",
    "note.beamedRight1-S-2",
    "note.beamedLeft3-L7",
    "note.thirty_second-S4",
    "note.beamedRight2-L-1",
    "note.whole-L7",
    "gracenote.eighth-L0",
    "note.whole-L6",
    "note.beamedBoth1-S7",
    "note.beamedBoth4-L3",
    "note.beamedRight3-S5",
    "note.sixteenth-L0",
    "gracenote.eighth-L1",
    "note.thirty_second-L2",
    "gracenote.thirty_second-L5",
    "note.beamedRight3-L0",
    "note.beamedLeft2-L-1",
    "note.beamedBoth3-L0",
    "note.double_whole-L2",
    "note.thirty_second-S3",
    "note.beamedBoth4-L5",
    "gracenote.sixteenth-S1",
    "note.beamedLeft2-S-2",
    "note.beamedRight3-L6",
    "gracenote.beamedBoth1-S4",
    "note.beamedRight3-S0",
    "note.beamedBoth4-L4",
    "note.beamedBoth4-S4",
    "note.thirty_second-L3",
    "note.double_whole-S3",
    "accidental.flat-S7",
    "note.beamedBoth3-L-1",
    "note.beamedLeft4-S5",
    "slur.start-S7",
    "slur.end-S7",
    "rest.sixty_fourth-L3",
    "gracenote.thirty_second-S5",
    "gracenote.thirty_second-L6",
    "note.eighth-L8",
    "note.beamedRight3-L1",
    "note.beamedBoth0-S4",
    "note.thirty_second-S2",
    "gracenote.sixteenth-L6",
    "note.thirty_second-L5",
    "accidental.natural-L0",
    "note.beamedBoth2-S-2",
    "note.beamedRight1-L8",
    "note.beamedLeft3-S7",
    "slur.start-L0",
    "slur.end-L0",
    "note.quarter-L8",
    "gracenote.beamedBoth1-L4",
    "note.beamedBoth4-S5",
    "gracenote.beamedBoth2-L4",
    "gracenote.beamedBoth1-S5",
    "note.beamedRight3-S7",
    "note.beamedBoth4-S3",
    "gracenote.thirty_second-S4",
    "note.half-L-1",
    "digit.16-L2",
    "note.eighth-S-2",
    "gracenote.thirty_second-L3",
    "note.whole-S6",
    "note.beamedBoth2-L-1",
    "note.eighth-S8",
    "gracenote.quarter-L4",
    "note.thirty_second-S5",
    "slur.start-S-1",
    "slur.end-S-1",
    "gracenote.beamedBoth2-S4",
    "note.beamedRight3-L7",
    "gracenote.sixteenth-L1",
    "note.beamedLeft0-L4",
    "gracenote.thirty_second-L4",
    "gracenote.quarter-S5",
    "accidental.sharp-L7",
    "note.double_whole-S4",
    "note.beamedRight0-S6",
    "note.beamedBoth2-L8",
    "note.thirty_second-L4",
    "accidental.natural-S0",
    "gracenote.quarter-L3",
    "gracenote.quarter-S2",
    "note.beamedRight0-S5",
    "clef.F-L3",
    "note.beamedLeft0-S5",
    "note.beamedBoth3-S7",
    "gracenote.eighth-S0",
    "gracenote.thirty_second-S0",
    "note.beamedLeft0-S3",
    "note.beamedRight2-S-2",
    "note.beamedLeft1-L8",
    "dot-S8",
    "note.beamedBoth4-S1",
    "note.beamedBoth4-L2",
    "note.beamedBoth4-S2",
    "note.beamedLeft4-S3",
    "note.beamedRight4-S2",
    "note.beamedRight2-L8",
    "accidental.flat-S0",
    "accidental.sharp-S6",
    "note.beamedRight0-L3",
    "note.beamedBoth0-L3",
    "note.half-L8",
    "note.beamedBoth0-S3",
    "accidental.natural-L7",
    "note.sixteenth-L7",
    "accidental.sharp-S-2",
    "note.double_whole-L3",
    "note.beamedLeft4-L2",
    "note.beamedRight0-S3",
    "gracenote.beamedBoth1-S3",
    "note.beamedBoth0-L4",
    "slur.start-L-1",
    "slur.end-L-1",
    "gracenote.beamedBoth2-S3",
    "note.beamedLeft4-L5",
    "note.beamedLeft4-L6",
    "note.beamedLeft4-S6",
    "gracenote.beamedBoth2-L5",
    "note.beamedBoth4-L1",
    "note.beamedLeft4-S2",
    "note.thirty_second-L1",
    "note.beamedBoth0-L2",
    "gracenote.beamedBoth1-L3",
    "gracenote.quarter-S4",
    "note.quarter-L-3",
    "note.double_whole-S2",
    "gracenote.beamedBoth2-S2",
    "note.sixteenth-L-1",
    "note.beamedLeft4-S4",
    "gracenote.thirty_second-S3",
    "dot-S-2",
    "gracenote.sixteenth-S0",
    "gracenote.thirty_second-L2",
    "note.thirty_second-L-1",
    "note.whole-S-1",
    "note.thirty_second-L6",
    "note.beamedLeft0-L5",
    "note.thirty_second-S6",
    "accidental.flat-L7",
    "gracenote.beamedBoth2-S5",
    "note.beamedRight3-S-2",
    "note.beamedLeft0-S2",
    "gracenote.beamedBoth1-S2",
    "gracenote.sixteenth-L-1",
    "note.thirty_second-S1",
    "note.beamedRight0-S2",
    "gracenote.beamedBoth1-L5",
    "gracenote.quarter-S3",
    "gracenote.beamedBoth3-L6",
    "note.beamedBoth0-S2",
    "note.beamedRight0-L5",
    "note.double_whole-S6",
    "note.eighth-L-2",
]

BLANK_IDX  = 0
VOCAB_SIZE = len(VOCAB)
SYM_TO_IDX = {s: i for i, s in enumerate(VOCAB)}
IDX_TO_SYM = {i: s for i, s in enumerate(VOCAB)}

# ── Model ─────────────────────────────────────────────────────────────────────
class ResNetBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


class CRNN(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, rnn_hidden=256, rnn_layers=2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            ResNetBlock(64),
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            ResNetBlock(128), ResNetBlock(128),
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1), (2, 1)),
            ResNetBlock(256), ResNetBlock(256),
            nn.Conv2d(256, 512, 3, padding=1, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1), (2, 1)),
            nn.AdaptiveAvgPool2d((1, None)),
        )
        self.rnn = nn.LSTM(
            input_size=512, hidden_size=rnn_hidden,
            num_layers=rnn_layers, batch_first=False,
            bidirectional=True,
            dropout=0.3 if rnn_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(rnn_hidden * 2, vocab_size)

    def forward(self, x):
        features = self.cnn(x)
        features = features.squeeze(2)
        features = features.permute(2, 0, 1)
        output, _ = self.rnn(features)
        output = self.fc(output)
        return F.log_softmax(output, dim=2)


# ── Greedy CTC decode ─────────────────────────────────────────────────────────
def _greedy_ctc_decode(log_probs):
    indices = torch.argmax(log_probs, dim=1).tolist()
    decoded, prev = [], None
    for idx in indices:
        if idx != BLANK_IDX and idx != prev:
            decoded.append(idx)
        prev = idx
    return decoded


# ── Symbol Error Rate ─────────────────────────────────────────────────────────
def compute_ser(predicted, ground_truth):
    n = len(ground_truth)
    if n == 0:
        return 0.0
    m = len(predicted)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ground_truth[i-1] == predicted[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[n][m] / n

print ("Done Initializing")

Done Initializing


In [ ]:
import os, glob, torch
from torch.utils.data import Dataset
import cv2, numpy as np
from torchvision import transforms

class CameraPrIMuSDataset(Dataset):
    def __init__(self, root_dir: str, vocab: dict,
                 img_height: int = 64, augment: bool = False,
                 distorted_prob: float = 0.5, force_distorted: bool = False):
        """distorted_prob: when augment=True, fraction of samples drawn from the
        _distorted.jpg camera-simulation variant instead of the clean .png render
        (falls back to clean if a sample has no distorted file on disk). Session
        measurement (2026-09-09): 0.70% SER on clean vs 18.76% SER on
        _distorted.jpg for a checkpoint that never trained on the distorted
        variant at all -- this is the primary lever for closing that gap.
        force_distorted: always use the distorted variant when it exists, with
        no random draw and no extra synthetic _augment() on top -- for a
        dedicated "distorted-only" validation pass, independent of augment.
        """
        self.vocab          = vocab          # SYM_TO_IDX from your stage3 file
        self.img_height      = img_height
        self.augment          = augment
        self.distorted_prob  = distorted_prob
        self.force_distorted = force_distorted
        self.samples          = []  # (clean_img_path, distorted_img_path_or_None, label_list)

        # Gather all sample folders
        for agnostic_path in glob.glob(
                f"{root_dir}/**/*.agnostic", recursive=True):
            img_path = agnostic_path.replace(".agnostic", ".png")
            if not os.path.exists(img_path):
                continue
            distorted_path = agnostic_path.replace(".agnostic", "_distorted.jpg")
            if not os.path.exists(distorted_path):
                distorted_path = None
            with open(agnostic_path) as f:
                raw_tokens = f.read().split()
            label = [vocab.get(t, vocab["<unk>"]) for t in raw_tokens]
            if len(label) > 0:
                self.samples.append((img_path, distorted_path, label))

        n_distorted = sum(1 for _, d, _ in self.samples if d is not None)
        print(f"Loaded {len(self.samples)} samples from {root_dir} "
              f"({n_distorted} have a _distorted.jpg camera-simulation variant)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, distorted_path, label = self.samples[idx]

        if self.force_distorted and distorted_path is not None:
            use_distorted = True
        elif self.augment and distorted_path is not None:
            use_distorted = np.random.rand() < self.distorted_prob
        else:
            use_distorted = False

        src_path = distorted_path if use_distorted else img_path
        img = cv2.imread(src_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            # _distorted.jpg occasionally fails to decode -- fall back to the clean render
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise RuntimeError(f"Could not read image: {img_path}")
        h, w = img.shape
        new_w = max(1, int(w * self.img_height / h))
        img = cv2.resize(img, (new_w, self.img_height),
                         interpolation=cv2.INTER_AREA)

        if self.augment:
            img = self._augment(img)

        img = img.astype(np.float32) / 255.0
        img = (img - 0.5) / 0.5
        tensor = torch.tensor(img).unsqueeze(0)   # (1, H, W)

        return tensor, torch.tensor(label, dtype=torch.long)

    def _augment(self, img: np.ndarray) -> np.ndarray:
        # Brightness/contrast jitter for camera robustness. Widened from the
        # original [0.8,1.2]/[-20,20] -- Camera-PrIMuS's own distortion pipeline
        # (Calvo-Zaragoza & Rizo, ISMIR 2018) lists "irregular lighting" as one
        # of its three named distortion categories, and the narrower range
        # under-covers the brightness/contrast swings phone auto-exposure
        # produces on a real photographed page.
        alpha = np.random.uniform(0.6, 1.4)  # contrast
        beta  = np.random.randint(-35, 35)   # brightness
        img   = np.clip(alpha * img + beta, 0, 255).astype(np.uint8)

        # Directional shadow / uneven-lighting gradient -- a uniform global
        # brightness/contrast jitter can't represent one side of a handheld
        # photo being darker than the other. Fills the gap between the paper's
        # "irregular lighting" category and the OCR/HTR "shadow/background
        # disturbance" technique, neither of which the uniform jitter above
        # covers on its own.
        if np.random.rand() < 0.3:
            img = self._add_shadow(img)

        # Small random rotation -- previously missing entirely. Camera-PrIMuS's
        # distortion pipeline explicitly names rotation as one of its three
        # distortion categories; horizontal stretch and perspective warp below
        # are different transforms and don't substitute for it. Kept to a small
        # +/-3 deg range since strips are already tightly cropped to one
        # staff's height (STRIP_HEIGHT=64 in stage2) -- a larger angle risks
        # rotating stafflines/noteheads out of frame.
        if np.random.rand() < 0.3:
            angle = np.random.uniform(-3.0, 3.0)
            img = self._rotate(img, angle)

        # Small random horizontal stretch (camera distortion sim)
        if np.random.rand() < 0.3:
            scale = np.random.uniform(0.9, 1.1)
            new_w = max(1, int(img.shape[1] * scale))
            img = cv2.resize(img, (new_w, img.shape[0]))

        # Blur -- defocus or handheld motion, camera photos are rarely
        # pixel-sharp. Kernel range widened to include 7 (previously capped at
        # 5): handheld motion blur is often stronger than mild defocus alone.
        if np.random.rand() < 0.35:
            k = int(np.random.choice([3, 5, 7]))
            if np.random.rand() < 0.5:
                img = cv2.GaussianBlur(img, (k, k), 0)
            else:
                kernel = np.zeros((k, k), dtype=np.float32)
                kernel[k // 2, :] = 1.0 / k
                img = cv2.filter2D(img, -1, kernel)

        # Random perspective warp -- page rarely perfectly frontal to camera
        if np.random.rand() < 0.3:
            img = self._random_perspective(img)

        # Elastic distortion -- paper curl / lens distortion. OCR/HTR
        # robustness literature reports this as the single highest-impact
        # augmentation for closing a clean/distorted accuracy gap, so
        # probability raised 0.2->0.35 here (intensity left unchanged --
        # alpha/sigma are already tuned to stay legible on thin stafflines;
        # going stronger risks destroying notehead shapes the CTC head
        # depends on).
        if np.random.rand() < 0.35:
            img = self._elastic_distort(img)

        return img

    def _rotate(self, img: np.ndarray, angle_deg: float) -> np.ndarray:
        h, w = img.shape
        matrix = cv2.getRotationMatrix2D((w / 2, h / 2), angle_deg, 1.0)
        return cv2.warpAffine(img, matrix, (w, h), borderValue=255,
                               flags=cv2.INTER_LINEAR)

    def _add_shadow(self, img: np.ndarray) -> np.ndarray:
        """Simulates uneven lighting with a linear darkening gradient across
        the strip, in a random direction and orientation -- approximates the
        shadow a handheld phone throws across part of a page, which the
        uniform brightness/contrast jitter above can't represent on its own.
        """
        h, w = img.shape
        if np.random.rand() < 0.5:
            ramp = np.linspace(0, 1, w, dtype=np.float32)
            if np.random.rand() < 0.5:
                ramp = ramp[::-1]
            gradient = np.tile(ramp, (h, 1))
        else:
            ramp = np.linspace(0, 1, h, dtype=np.float32)
            if np.random.rand() < 0.5:
                ramp = ramp[::-1]
            gradient = np.tile(ramp.reshape(-1, 1), (1, w))
        strength = np.random.uniform(0.15, 0.45)  # max darkening fraction
        darken = 1.0 - gradient * strength
        shadowed = img.astype(np.float32) * darken
        return np.clip(shadowed, 0, 255).astype(np.uint8)

    def _random_perspective(self, img: np.ndarray) -> np.ndarray:
        h, w = img.shape
        jitter = 0.04  # max corner displacement, as a fraction of each dimension
        src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
        offsets = np.random.uniform(-jitter, jitter, src.shape) * np.array([w, h])
        dst = (src + offsets).astype(np.float32)
        matrix = cv2.getPerspectiveTransform(src, dst)
        return cv2.warpPerspective(img, matrix, (w, h), borderValue=255)

    def _elastic_distort(self, img: np.ndarray, alpha: float = 6.0, sigma: float = 4.0) -> np.ndarray:
        h, w = img.shape
        dx = cv2.GaussianBlur((np.random.rand(h, w).astype(np.float32) * 2 - 1), (0, 0), sigma) * alpha
        dy = cv2.GaussianBlur((np.random.rand(h, w).astype(np.float32) * 2 - 1), (0, 0), sigma) * alpha
        x, y = np.meshgrid(np.arange(w), np.arange(h))
        map_x = (x + dx).astype(np.float32)
        map_y = (y + dy).astype(np.float32)
        return cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderValue=255)

In [5]:
def ctc_collate(batch):
    """Pad images to same width; pack labels for CTC loss."""
    images, labels = zip(*batch)

    # Pad images to max width in batch
    max_w = max(img.shape[2] for img in images)
    padded = torch.zeros(len(images), 1, images[0].shape[1], max_w)
    for i, img in enumerate(images):
        padded[i, :, :, :img.shape[2]] = img

    # Input lengths = time steps after CNN (width // 4 due to pooling)
    # Your CNN applies 2 MaxPool2d(2,2) on width and 2 MaxPool2d((2,1))
    # So width is halved twice: T = W // 4
    input_lengths = torch.tensor(
        [img.shape[2] // 4 for img in images], dtype=torch.long)

    # Flatten labels and record lengths
    label_lengths = torch.tensor(
        [len(lb) for lb in labels], dtype=torch.long)
    flat_labels = torch.cat(labels)

    return padded, flat_labels, input_lengths, label_lengths

In [ ]:
import os, time
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

def _evaluate_ser(model, loader, device):
    """Mean SER over one validation loader. Its own function so the same
    eval logic can run twice per epoch (clean vs. distorted variant) without
    duplicating the CTC-decode/SER-alignment logic."""
    model.eval()
    all_ser = []
    with torch.no_grad():
        for images, labels, in_lens, lb_lens in loader:
            images    = images.to(device)
            log_probs = model(images)   # (T, B, V)

            label_offset = 0
            for b in range(images.size(0)):
                lp       = log_probs[:, b, :]
                pred_ids = _greedy_ctc_decode(lp)

                gt_len = lb_lens[b].item()
                gt_ids = labels[label_offset : label_offset + gt_len].tolist()
                label_offset += gt_len

                pred_syms = [IDX_TO_SYM.get(i, "<unk>") for i in pred_ids]
                gt_syms   = [IDX_TO_SYM.get(i, "<unk>") for i in gt_ids]
                all_ser.append(compute_ser(pred_syms, gt_syms))
    return float(np.mean(all_ser)) if all_ser else 0.0


def train(root_dir="/kaggle/input/datasets/piamac/cameraprimus/CameraPrIMuS - Copy",
          weights_out="crnn_weights/best.pt",
          img_height=64, epochs=50, batch_size=16, lr=3e-4,
          distorted_prob=0.5):
    """distorted_prob: fraction of augmented training samples drawn from the
    _distorted.jpg camera-simulation variant instead of the clean .png render
    -- see CameraPrIMuSDataset's docstring for the measurement that motivates
    this default.
    """

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # -- Data -- separate dataset instances so augment/eval-mode doesn't bleed over.
    # All three instances glob the same root_dir in the same (deterministic) order,
    # so a given index refers to the same underlying sample across all of them --
    # the same assumption the original train/val split already relied on.
    full_train_ds         = CameraPrIMuSDataset(root_dir, SYM_TO_IDX, img_height=img_height,
                                                 augment=True, distorted_prob=distorted_prob)
    full_val_clean_ds     = CameraPrIMuSDataset(root_dir, SYM_TO_IDX, img_height=img_height,
                                                 augment=False)
    full_val_distorted_ds = CameraPrIMuSDataset(root_dir, SYM_TO_IDX, img_height=img_height,
                                                 augment=False, force_distorted=True)

    # Split indices -- same seed so train/val never overlap
    indices  = list(range(len(full_train_ds)))
    val_size = int(0.1 * len(indices))
    train_indices = indices[val_size:]   # 90%
    val_indices   = indices[:val_size]   # 10%

    train_ds         = Subset(full_train_ds,         train_indices)
    val_ds_clean      = Subset(full_val_clean_ds,      val_indices)
    val_ds_distorted  = Subset(full_val_distorted_ds,  val_indices)

    print(f"Train: {len(train_ds):,} samples | "
          f"Val: {len(val_ds_clean):,} samples (evaluated clean AND distorted)")

    train_loader        = DataLoader(train_ds, batch_size=batch_size,
                                     shuffle=True,  collate_fn=ctc_collate,
                                     num_workers=0, pin_memory=True)
    val_loader_clean     = DataLoader(val_ds_clean, batch_size=batch_size,
                                     shuffle=False, collate_fn=ctc_collate,
                                     num_workers=0, pin_memory=True)
    val_loader_distorted = DataLoader(val_ds_distorted, batch_size=batch_size,
                                     shuffle=False, collate_fn=ctc_collate,
                                     num_workers=0, pin_memory=True)

    # -- Model
    model     = CRNN(VOCAB_SIZE).to(device)
    criterion = torch.nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # "Best" checkpoint is selected on DISTORTED SER, not clean -- that's the
    # metric that matches real (camera-photo) deployment. clean_ser is tracked
    # and printed alongside it purely as a canary: if it starts climbing while
    # distorted_ser falls, fine-tuning is trading away clean-image accuracy for
    # distortion robustness (the failure mode documented in arXiv 2202.10054),
    # and that trade-off should be a deliberate call, not something silent.
    best_distorted_ser = float("inf")
    start_epoch = 1
    os.makedirs("crnn_weights", exist_ok=True)

    # -- Resume from checkpoint if it exists --------------------------------
    if os.path.exists(weights_out):
        checkpoint = torch.load(weights_out, map_location=device)
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        best_distorted_ser = checkpoint.get("distorted_ser", checkpoint.get("ser", float("inf")))
        start_epoch = checkpoint["epoch"] + 1
        print(f"  ⇺ Resumed from epoch {checkpoint['epoch']} "
              f"(best distorted SER={best_distorted_ser:.4f})")

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time()

        # -- Train loop
        model.train()
        train_loss = 0.0
        for images, labels, in_lens, lb_lens in train_loader:
            images  = images.to(device)
            labels  = labels.to(device)
            in_lens = in_lens.to(device)
            lb_lens = lb_lens.to(device)

            log_probs = model(images)                          # (T, B, V)
            in_lens   = in_lens.clamp(max=log_probs.shape[0]) # safety clamp

            loss = criterion(log_probs, labels, in_lens, lb_lens)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += loss.item()

        scheduler.step()

        # -- Validation -- clean and distorted tracked separately (see docstring above)
        clean_ser     = _evaluate_ser(model, val_loader_clean, device)
        distorted_ser = _evaluate_ser(model, val_loader_distorted, device)
        epoch_time    = time.time() - t0

        print(f"Epoch {epoch:03d} | "
              f"loss {train_loss / len(train_loader):.4f} | "
              f"clean SER {clean_ser:.4f} | "
              f"distorted SER {distorted_ser:.4f} | "
              f"{epoch_time:.1f}s")

        if distorted_ser < best_distorted_ser:
            best_distorted_ser = distorted_ser
            torch.save({
                "model_state":     model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "epoch":           epoch,
                "distorted_ser":   best_distorted_ser,
                "clean_ser":       clean_ser,
                "ser":             best_distorted_ser,  # backward-compat key for any external reader expecting "ser"
                "vocab":           VOCAB,
            }, weights_out)
            print(f"  ✓ Saved best model (distorted SER={best_distorted_ser:.4f}, "
                  f"clean SER={clean_ser:.4f}) → {weights_out}")

    print(f"\nTraining complete. Best distorted SER: {best_distorted_ser:.4f}")

train()

In [1]:
# Check if clean PrIMuS has any new tokens not in current VOCAB
import collections
from pathlib import Path

current_vocab = set(VOCAB)
new_tokens = collections.Counter()

for f in Path("/kaggle/input/datasets/umerellous/primus/primus").rglob("*.agnostic"):
    with open(f) as file:
        for token in file.read().split():
            if token not in current_vocab:
                new_tokens[token] += 1

print(f"New tokens not in current VOCAB: {len(new_tokens)}")
for token, count in new_tokens.most_common(20):
    print(f"  {token}: {count}")

NameError: name 'VOCAB' is not defined